# Lab 25-08 · Spreading, Fins, Cold Plates and Mission Profiles

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import thermal as th

## Part 1 · Spreading

*Predict*: does the 45° estimate get better or worse as the lid thickens?

In [2]:
a, b, k = 16e-3, 35e-3, 400
for t in (1e-3, 3e-3, 5e-3, 10e-3):
    fd = th.spreader_fd(a, b, t, k, 160, max(24, int(t / 1e-3 * 16)))
    r45 = th.r45_disk(a, t, k, b)
    print(f"t = {t*1e3:4.1f} mm: exact {fd*1e3:6.3f} mK/W, 45 deg {r45*1e3:6.3f} mK/W, error {100*(r45-fd)/fd:5.1f} %")
coarse = th.spreader_fd(a, b, 3e-3, k, 80, 24)
fine = th.spreader_fd(a, b, 3e-3, k, 160, 48)
print(f"grid check: {coarse*1e3:.4f} vs {fine*1e3:.4f} mK/W")

t =  1.0 mm: exact  3.003 mK/W, 45 deg  2.926 mK/W, error  -2.6 %
t =  3.0 mm: exact  8.381 mK/W, 45 deg  7.853 mK/W, error  -6.3 %
t =  5.0 mm: exact 12.945 mK/W, 45 deg 11.842 mK/W, error  -8.5 %


t = 10.0 mm: exact 21.259 mK/W, 45 deg 19.129 mK/W, error -10.0 %
grid check: 8.3715 vs 8.3813 mK/W


## Part 2 · Fins

*Predict* the efficiency of the copper fin.

In [3]:
eta_f = th.fin_efficiency(50, 400, 1e-3, 0.04)
eta_n = th.fin_efficiency_ode(50, 400, 1e-3, 0.04)
print(f"formula {eta_f:.4f}, numerical {eta_n:.4f}")
ml = np.linspace(0.05, 4, 100)
plt.figure()
plt.plot(ml, np.tanh(ml) / ml)
plt.xlabel("mL"); plt.ylabel("fin efficiency"); plt.grid(True)
plt.show()

formula 0.8850, numerical 0.8850


## Part 3 · Cold-plate limit

*Predict* the largest power at 0.035 K/W, 45 °C inlet and 2 L/min.

In [4]:
def limit(flow_lpm, r=0.035, t_in=45, t_max=105):
    return optimize.brentq(lambda p: th.junction(p, t_in, flow_lpm / 6e4, r) - t_max, 1, 1e5)
print(f"limit at 2 L/min: {limit(2):.0f} W")
flows = np.linspace(0.5, 10, 60)
plt.figure()
plt.plot(flows, [limit(f) for f in flows])
plt.xlabel("coolant flow (L/min)"); plt.ylabel("power limit (W)"); plt.grid(True)
plt.show()

limit at 2 L/min: 1555 W


## Part 4 · Mission profile

*Predict* the Arrhenius-equivalent temperature at 1.0 eV.

In [5]:
temps, w = [-40, 70, 110, 125], [0.05, 0.65, 0.20, 0.10]
print(f"1.0 eV: {th.arrhenius_equivalent(temps, w, 1.0):.1f} C")
eas = np.linspace(0.2, 1.5, 40)
plt.figure()
plt.plot(eas, [th.arrhenius_equivalent(temps, w, e) for e in eas])
plt.axhline(78, ls=":", color="k")
plt.xlabel("activation energy (eV)"); plt.ylabel("equivalent temperature (C)"); plt.grid(True)
plt.show()

1.0 eV: 102.2 C
